# Comet benchmark: Iceberg -> Iceberg

Creates Iceberg source tables, runs the queries with Comet writing results to Iceberg tables (AWS Glue catalog), then disables Comet and repeats the same queries.

## 0. Start the Spark session

In [ ]:
import os  # belt-and-suspenders: clear any insecure Py4J hints
os.environ.pop("PYSPARK_GATEWAY_PORT", None)
os.environ.pop("PYSPARK_GATEWAY_SECRET", None)

try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession

COMET_JAR = "/opt/spark/jars/comet-spark-spark4.0_2.13-0.16.0.jar"

# Iceberg 1.10.0 runtime + AWS bundle, resolved from Maven at session start
ICEBERG_PACKAGES = (
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
    "org.apache.iceberg:iceberg-aws-bundle:1.10.0"
)

# ----------------- EDIT THESE -----------------
BUCKET     = "<<YOUR-S3-BUCKET>>"                 # bucket name only, e.g. my-bucket
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
WAREHOUSE  = f"s3://{BUCKET}/iceberg-warehouse"   # Iceberg/S3FileIO uses the s3:// scheme
# ----------------------------------------------

spark = (
    SparkSession.builder
      .master("spark://spark-master:7077")
      .appName("comet-iceberg-to-iceberg")
      .config("spark.jars.packages", ICEBERG_PACKAGES)
      # ---- Apache DataFusion Comet ----
      .config("spark.plugins", "org.apache.spark.CometPlugin")
      .config("spark.comet.enabled", "true")
      .config("spark.comet.exec.enabled", "true")
      .config("spark.comet.exec.shuffle.enabled", "true")
      .config("spark.shuffle.manager", "org.apache.spark.sql.comet.execution.shuffle.CometShuffleManager")
      .config("spark.comet.explainFallback.enabled", "true")
      .config("spark.memory.offHeap.enabled", "true")
      .config("spark.memory.offHeap.size", "8g")
      .config("spark.comet.exec.memoryPool", "fair_unified")
      .config("spark.comet.exec.memoryPool.fraction", "0.7")
      .config("spark.executorEnv.COMET_WORKER_THREADS", "4")
      .config("spark.driver.extraClassPath", COMET_JAR)
      .config("spark.executor.extraClassPath", COMET_JAR)
      # ---- Apache Iceberg + AWS Glue catalog ----
      .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
      .config("spark.sql.catalog.glue", "org.apache.iceberg.spark.SparkCatalog")
      .config("spark.sql.catalog.glue.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
      .config("spark.sql.catalog.glue.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
      .config("spark.sql.catalog.glue.warehouse", WAREHOUSE)
      .config("spark.sql.catalog.glue.client.region", AWS_REGION)
      .config("spark.sql.catalog.glue.glue.skip-name-validation", "true")
      .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
      .config("spark.hadoop.fs.s3a.bucket.all.committer.magic.enabled", "true")
      .config("spark.eventLog.enabled", "false")
      .config("spark.dynamicAllocation.enabled", "false")
      .config("spark.log.level", "warn")
      .getOrCreate()
)
print("Spark", spark.version, "ready | warehouse:", WAREHOUSE, "| region:", AWS_REGION)

## 1. Parameters (identical across the three notebooks)

In [ ]:
from pyspark.sql import functions as F
import time

# --------- Data tunables (identical across all three notebooks) ----------
N_ROWS = 50_000_000
N_PROD = 50_000
N_CTRY = 32
N_CH   = 6
N_DAYS = 90
REPART = 256
BUCKETS_FOR_PCTS = 2048
SEED = 42   # fixed seeds -> reproducible, identical data across the three notebooks

# --------- S3 locations / catalog ----------
s3_base     = "<<YOUR-S3-BUCKET>>"          # e.g. s3a://my-bucket/comet-bench
PARQUET_DIR = f"{s3_base}/dataset"          # Parquet source location
OUT_DIR     = f"{s3_base}/results"          # Parquet result location (Parquet sink only)
CATALOG = "glue"
DB      = "comet-tests"
FQDB    = f"{CATALOG}.`{DB}`"               # backticks: the database name has a hyphen
print("Parquet dir:", PARQUET_DIR, "| Iceberg db:", FQDB)

## 2. Generate the synthetic data (identical across the three notebooks)

In [ ]:
# Synthetic, skewed source data registered as temp views (identical in every notebook).
# Fixed seeds make the data reproducible so all three notebooks operate on the same rows.
(spark.range(N_ROWS)
   .select(
       (F.floor(F.pow(F.rand(SEED)*0.999999 + 1e-9, 3) * N_PROD)).alias("product_id"),
       (F.floor(F.pow(F.rand(SEED+1)*0.999999 + 1e-9, 1.7) * N_CTRY)).alias("country_id"),
       (F.floor(F.pow(F.rand(SEED+2)*0.999999 + 1e-9, 1.4) * N_CH)).alias("channel_id"),
       (F.floor(F.rand(SEED+3) * N_DAYS)).alias("day_idx"),
       (F.round(F.rand(SEED+4)*90 + 10, 2)).alias("price"),
       (F.floor(F.rand(SEED+5)*5 + 1)).alias("qty"),
       (F.round(F.rand(SEED+6)*0.25, 4)).alias("discount"))
   .withColumn("revenue", F.col("price") * F.col("qty") * (1 - F.col("discount")))
   .repartition(REPART)
   .createOrReplaceTempView("gen_fact"))

(spark.range(N_CTRY)
   .select(F.col("id").alias("country_id"),
           F.concat(F.lit("C"), F.lpad(F.col("id").cast("string"), 2, "0")).alias("country_name"),
           (F.rand(SEED+7)*0.25 + 0.75).alias("vat"))
   .createOrReplaceTempView("gen_countries"))

(spark.range(N_CH)
   .select(F.col("id").alias("channel_id"),
           F.array(F.lit("web"), F.lit("store"), F.lit("partner"),
                   F.lit("marketplace"), F.lit("app"), F.lit("phone")).getItem(F.col("id")).alias("channel_name"))
   .createOrReplaceTempView("gen_channels"))

(spark.range(N_DAYS)
   .select(F.col("id").cast("int").alias("day_idx"),
           F.date_add(F.current_date(), F.lit(-N_DAYS) + F.col("id").cast("int")).alias("date_key"))
   .createOrReplaceTempView("gen_calendar"))

TABLES = ["fact", "countries", "channels", "calendar"]
print("Synthetic source views ready:", [f"gen_{t}" for t in TABLES])

## 3. Materialize the source and register query views

Reads and writes Iceberg tables via the Glue catalog (`s3://` warehouse).

In [ ]:
# Create the Glue database and the Iceberg SOURCE tables (idempotent via IF NOT EXISTS)
spark.sql(f"CREATE DATABASE IF NOT EXISTS {FQDB}")
for t in TABLES:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {FQDB}.src_{t} USING iceberg AS SELECT * FROM gen_{t}")

# Register the Iceberg sources as the query temp views
for t in TABLES:
    spark.read.table(f"{FQDB}.src_{t}").createOrReplaceTempView(t)

spark.sql(f"SHOW TABLES IN {FQDB}").show(truncate=False)
print("Iceberg source tables ready in", FQDB)

## 4. Define the queries (identical across the three notebooks)

In [ ]:
# The three queries are identical across notebooks. They read the temp views
# `fact`, `countries`, `channels`, `calendar` (registered from each notebook's source).
sql_A = f"""
SELECT country_id, channel_id,
       count(*)                                              AS n_rows,
       sum(qty)                                              AS sum_qty,
       sum(revenue)                                          AS sum_revenue,
       avg(price)                                            AS avg_price,
       stddev_pop(price)                                     AS std_price,
       approx_count_distinct(product_id)                     AS card_products,
       sum(CASE WHEN discount > 0 THEN 1 ELSE 0 END)         AS num_disc,
       percentile_approx(price, array(0.25,0.5,0.75), {BUCKETS_FOR_PCTS}) AS price_quartiles
FROM fact
GROUP BY country_id, channel_id
ORDER BY sum_revenue DESC
"""

sql_B = """
SELECT c.date_key, f.country_id, f.product_id,
       sum(f.revenue)  AS rev,
       sum(f.qty)      AS qty,
       avg(f.discount) AS avg_disc
FROM fact f
JOIN calendar c ON f.day_idx = c.day_idx
GROUP BY ROLLUP(c.date_key, f.country_id, f.product_id)
HAVING sum(f.revenue) IS NOT NULL
ORDER BY rev DESC
"""

sql_C = """
SELECT /*+ BROADCAST(co), BROADCAST(ch) */
       co.country_name, ch.channel_name, ca.date_key,
       sum(f.revenue)               AS gross,
       sum(f.revenue / co.vat)      AS net,
       count(DISTINCT f.product_id) AS sku_seen
FROM fact f
JOIN countries co ON f.country_id = co.country_id
JOIN channels  ch ON f.channel_id = ch.channel_id
JOIN calendar  ca ON f.day_idx    = ca.day_idx
GROUP BY co.country_name, ch.channel_name, ca.date_key
ORDER BY gross DESC
LIMIT 5000
"""
QUERIES = {"qa_result": sql_A, "qb_result": sql_B, "qc_result": sql_C}
print("Queries defined:", list(QUERIES))

## 5. Sink helper + Comet toggle

In [ ]:
def run_query(label, sql):
    """Iceberg sink: run the SELECT and materialize the result into an Iceberg table."""
    target = f"{FQDB}.{label}"
    start = time.time()
    spark.sql(f"CREATE OR REPLACE TABLE {target} USING iceberg AS {sql}")
    rows = spark.sql(f"SELECT count(*) AS c FROM {target}").first()["c"]
    secs = time.time() - start
    print(f"  {label}: {rows:,} rows -> {secs:.2f}s")
    return secs

In [ ]:
def set_comet(on: bool):
    """Toggle Comet execution + shuffle at runtime (the shuffle MANAGER stays static)."""
    flag = str(bool(on)).lower()
    spark.conf.set("spark.comet.enabled", flag)
    spark.conf.set("spark.comet.exec.enabled", flag)
    spark.conf.set("spark.comet.exec.shuffle.enabled", flag)
    return spark.conf.get("spark.comet.enabled")

## 6. Run with Comet ON, then OFF, and compare

In [ ]:
def run_suite(tag):
    print(f"=== run: {tag} (comet={spark.conf.get('spark.comet.enabled')}) ===")
    times = {}
    for label, sql in QUERIES.items():
        times[label] = run_query(label, sql)
    return times

# 1) Comet ON, 2) Comet OFF -- same queries, same data, same sinks
set_comet(True)
on = run_suite("comet-on")
set_comet(False)
off = run_suite("comet-off")

print("\n              Comet ON    Comet OFF   speedup")
for k in QUERIES:
    print(f"{k:>9}:  {on[k]:8.2f}s  {off[k]:9.2f}s  {off[k]/on[k]:6.2f}x")
# restore Comet for any further interactive work
set_comet(True)